[< Back to Main README](../README.md) | [Demo README](./README.md)

# Semantic Tool Selection with a Neo4j Tool Graph

Based on: [Internal Representations as Indicators of Hallucinations in Agent Tool Selection](https://arxiv.org/pdf/2601.05214)

## What This Demo Shows

An agent with 31 travel tools pays for every tool schema on every call, and a large
tool pool raises the chance of a wrong tool selection. This demo stores the tools in
Neo4j, selects the top 3 per question with vector search, and explains each selection
using the graph relationships around the candidates.

The live path is three representative questions. For each one the notebook prints:

1. The **vector candidates** with their similarity scores.
2. The **graph-expanded tools**, each with the relationship that caused its inclusion.

The full 24-query Traditional versus Semantic versus Semantic+Memory measurement
lives in `maintainer_harness.py`. It is maintainer-only validation, and the token
cost section near the end of this notebook quotes its measured numbers.

### Where the tools live

Each tool is a `:Tool` node in the same Neo4j Aura instance as the Demo 01 hotel
knowledge graph. The node carries the tool's description and a Nova 2 description
embedding, and the `tool_description_embeddings` vector index serves the top-k
selection. Tools are also linked to the `:Concept` nodes they require or produce,
which the live questions below use for workflow-aware expansion.

The pool is 31 tools: the 29 mock travel tools plus the two real-data hotel tools,
which always run against the knowledge graph now that Neo4j is a requirement of
this demo.

### The cost being attacked

Every tool schema is sent on every call. The maintainer harness measures roughly
6,500 tokens per query for the Traditional variant, counting the tool schemas plus
the system prompt, the user turn, tool results, and model output. Filtering to the
top 3 tools removes most of the schema cost while leaving everything else unchanged.

### The Pipeline

```
User Query → Neo4j Vector Index → Top 3 Tools → Graph Expansion → Agent → Tool Call
```


## Important: Execution Order

**This notebook must be executed sequentially from top to bottom.**

To run correctly:
1. Click **"Run All"** in the Jupyter menu, OR
2. Execute each cell in order using **Shift+Enter**

Each live question depends on the setup and tool-graph build cells above it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# This demo is Bedrock-only and needs no other provider credentials.
# To swap providers, see:
#   https://strandsagents.com/docs/user-guide/concepts/model-providers/

## Setup

In [ ]:
from strands import Agent
# Model configuration: Amazon Bedrock (default, no extra import needed).
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use a specific Bedrock model:
#   MODEL = "us.anthropic.claude-sonnet-5"
#   agent = Agent(tools=..., model=MODEL)
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/
from enhanced_tools import ALL_TOOLS
from registry import build_index, search_tools, select_tools_with_context

PROMPT = "You are a travel assistant. Use the correct tool to answer questions."

print(f"✅ Loaded {len(ALL_TOOLS)} tools")

## Build the Neo4j Tool Graph

`build_index` writes each tool as a `:Tool` node with its Nova 2 description
embedding, links it to the `:Concept` nodes it requires or produces, groups tools
into `:Domain` nodes, and creates the `tool_description_embeddings` vector index.
The build is idempotent: re-running it replaces the tool graph in place.

This requires the Neo4j connection configured for Demo 01. If the build fails,
check `01-graphrag-demo/.env` and run `uv run 01-graphrag-demo/prepare_graph.py`.

In [ ]:
build_index(ALL_TOOLS)
print("✅ Neo4j tool graph built")

## Three Live Questions

Vector search ranks tools by description similarity, and that is all a standalone
vector store can do. Because the tools live in a graph, one more query shape becomes
available: expand from the vector candidates along the `REQUIRES` and `PRODUCES`
relationships to the tools nearby in the workflow.

- A **downstream** tool consumes a concept a candidate produces: `book_hotel`
  produces a `booking`, and `process_payment` requires one.
- An **upstream** tool produces a concept a candidate requires: `book_hotel`
  requires `hotel_availability`, which `check_hotel_availability` produces.

Each expanded tool comes with the relationship that caused its inclusion, so the
selection is explainable rather than a bare similarity score.

**Read this as workflow-aware discovery and explainability, not as an accuracy
fix.** Expansion does not repair a question whose correct tool fell outside the
vector top-k; it surfaces the tools an agent will plausibly need next and says why.

The three questions below cover a hotel search, a booking in the middle of a
workflow, and a cancellation. Their top-3 candidates and similarity scores are
stable across repeated runs against the live graph, so they are safe to run in
front of an audience.

In [ ]:
def show_selection(query: str) -> None:
    """Print vector candidates and graph-expanded tools for one question."""
    report = select_tools_with_context(query, top_k=3)
    print("=" * 80)
    print(f"Query: {query}")
    print("\n  Vector candidates (similarity signal):")
    for candidate in report["candidates"]:
        print(f"    {candidate['name']:<32} {candidate['reason']}")
    print("\n  Graph expansion (workflow relationships):")
    if not report["expanded"]:
        print("    (none)")
    for tool in report["expanded"]:
        print(f"    {tool['name']:<32} {tool['reason']}")

### Question 1: "Find real hotels in France"

A hotel search. `search_real_hotels`, the tool that queries the Demo 01 knowledge
graph, wins on similarity, with the mock `search_hotels` close behind. Every
expansion here is **downstream**: a search produces a `hotel`, and the graph lists
the tools that consume one, from reviews and pricing through availability to
booking. The expansion reads as the natural next steps after a search.

In [ ]:
show_selection("Find real hotels in France")

### Question 2: "Book AnyCompany Hotel for John Smith"

A booking request in the middle of a workflow. `book_hotel` wins on similarity, and
the expansion works in both directions:

- **Upstream:** `check_hotel_availability` and `check_hotel_availability_dates`
  produce the `hotel_availability` that `book_hotel` requires. The graph is saying
  that a booking should be preceded by an availability check.
- **Downstream:** `process_payment` requires the `booking` that `book_hotel`
  produces. The graph is saying what comes next after the booking succeeds.

This is the query shape a bare vector store cannot answer: not "what looks like
this question" but "what belongs to this workflow, and why".

In [ ]:
show_selection("Book AnyCompany Hotel for John Smith")

### Question 3: "Cancel reservation BK-67890"

A cancellation and refund request. The generic `cancel` tool and `refund_payment`
lead on similarity. The expansion explains the refund path in both directions:
`refund_payment` requires a `payment`, which `process_payment` produces, and
`cancel` requires a `booking`, which `book_hotel` and `book_flight` produce. The
graph is saying that a refund only makes sense downstream of a payment, and a
cancellation only makes sense downstream of a booking.

In [ ]:
show_selection("Cancel reservation BK-67890")

## One Agent Call with the Selected Tools

The selection feeds a Strands agent. One bounded call closes the loop: filter the
first question to its top 3 tools, hand the agent only those, and watch it call the
real-data tool against the Demo 01 knowledge graph. The agent sees 3 tool schemas
instead of 31, which is where the token saving comes from.

In [ ]:
QUESTION = "Find real hotels in France"

selected = search_tools(QUESTION, top_k=3)
print(f"Tools sent to the agent: {[t.__name__ for t in selected]}")
print()

agent = Agent(tools=selected, system_prompt=PROMPT)
result = agent(QUESTION)

called = list(result.metrics.tool_metrics.keys()) if result.metrics else []
print(f"\n\nTool the agent called: {called}")

## The Token Cost Story (Maintainer-Measured)

The reason to filter tools at all is cost. The full measurement runs 24 queries
through three variants and scores each tool call against ground truth:

| Variant | Tools sent per query |
|---|---|
| Traditional | all 31 |
| Semantic | top 3, selected by Neo4j vector search |
| Semantic + Memory | top 3, plus conversation history bounded to 3 turns |

That evaluation makes roughly 72 live agent calls, so it does not run in this
notebook. It lives in **`maintainer_harness.py`**, which is **maintainer-only
validation**, not part of the workshop path:

```bash
uv run maintainer_harness.py
```

**Measured on a full harness run against the 31-tool pool:**

| Measured over 24 queries | Traditional | Semantic | Semantic + Memory |
|---|---|---|---|
| Total tokens | 162,719 | 40,647 | 78,769 |
| Avg tokens per query | 6,780 | 1,694 | 3,282 |
| Token reduction | baseline | **75.0%** | 51.6% |
| Tool selection accuracy | 18/24 | 17/24 | 18/24 |

These figures are LLM output and move between runs. Treat the semantic reduction
as a range of roughly 60-75% rather than a fixed number.

**Read the accuracy result carefully.** The headline result is the token reduction,
which is large and reproducible. Accuracy is measured because it is what a cost
optimization is most likely to damage, not because filtering is expected to improve
it. An accuracy gap of one or two queries out of 24 is noise, not a finding in
either direction. Filtering can only help if the correct tool survives the top-3
cut; when it does not, the agent cannot recover, and the harness's error analysis
labels exactly those cases.

---
## Summary

### What Semantic Tool Selection Buys You

1. **A large, reproducible token reduction.** Sending 3 tool schemas instead of 31
   is a constant saving on every call. This is the result the maintainer harness
   establishes and the reason to adopt the technique.
2. **Accuracy is a tradeoff to watch, not a win to claim.** Filtering can only help
   if the correct tool survives the top-3 cut. When it does not, the agent cannot
   recover, and the harness's error analysis labels exactly those cases. Measure
   accuracy on your own tool set and query mix before adopting this; do not assume
   the reduction comes for free.
3. **Explainable, workflow-aware selection.** Because the tools live in a Neo4j
   graph next to the concepts they require and produce, the selection names the
   relationship that pulled each tool in, and expansion surfaces the workflow steps
   around the vector candidates. The three live questions above show exactly this.

### Production Note: Swapping Tools Without Losing Memory

Strands supports dynamic tool swapping on a live agent. No agent recreation, no
conversation loss:

```python
# One agent, dynamic tools
agent = Agent(tools=initial_tools)

for query in queries:
    selected = search_tools(query, top_k=3)   # Neo4j vector index → top 3
    swap_tools(agent, selected)                # swap registry, keep memory
    agent(query)                               # conversation history preserved
```

`swap_tools()` works because Strands calls `tool_registry.get_all_tools_config()`
at each event loop cycle, so tool changes are picked up without restarting the
agent. The Semantic+Memory variant in `maintainer_harness.py` measures this
pattern, with conversation history bounded to 3 turns via `trim_history()`.
Without a bound, the resent transcript grows quadratically with turn count and
overtakes the saving from filtering tools.

### Reading the numbers

Everything an agent prints is LLM output and moves between runs. Treat the token
reduction as a range rather than a fixed figure, and treat any accuracy gap of one
or two queries out of 24 as noise until a larger evaluation says otherwise.

---

## References

### Research
- [Internal Representations as Indicators of Hallucinations in Agent Tool Selection](https://arxiv.org/pdf/2601.05214)

### Neo4j
- [Vector indexes](https://neo4j.com/docs/cypher-manual/current/indexes/semantic-indexes/vector-indexes/): the index type behind `tool_description_embeddings`

### Strands Agents
- [Strands Tool Registry](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/): dynamic tool management
- [Creating Custom Tools](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/): the `@tool` decorator
- [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/): swap to Amazon Bedrock, Anthropic, Ollama
- [Strands Agents Documentation](https://strandsagents.com): full framework docs

### Code
- [Code Repository](https://github.com/aws-samples/sample-stop-ai-agent-hallucinations-workshop)